In [ ]:
import json
z = json.load(open('/home/nte5cp/bsky_reposts/jortsthecat.bsky.social.json'))

In [11]:
import random, string, json
import datetime as dt

def make_random_string(length):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

def make_random_date_str(range_low, range_high, utc_z=True, return_datetime=False):
    delta = (range_high - range_low).total_seconds()
    delta = random.uniform(0, delta)
    new_time = range_low + dt.timedelta(seconds=delta)
    if utc_z == True:
        new_time_str = new_time.strftime('%Y-%m-%dT%H:%M:%S.%f')[:-3] + 'Z'
    elif utc_z == False:
        new_time_str = new_time.strftime('%Y-%m-%dT%H:%M:%S.%f')
    elif utc_z is None:
        new_time_str = new_time.strftime('%Y-%m-%dT%H:%M:%S') + '+07:00'

    if return_datetime:
        return new_time_str, new_time
    else:
        return new_time_str

In [27]:
A = 'a'
ab_did = A * 24
reposted_by_ab = [(A + B) * 12 for B in 'abc']
date_low = dt.datetime(year=2023, month=3, day=1, microsecond=0)
date_high = dt.datetime(year=2025, month=12, day=15, microsecond=0)

json_lines = []
for acct in reposted_by_ab:
    for i in range(2):
        repost_id = make_random_string(13)
        repost_uri = f'at://did:plc:{ab_did}/app.bsky.feed.repost/{repost_id}'
        created_at = make_random_date_str(date_low, date_high)
        op_id = make_random_string(13)
        reposted = f'at://did:plc:{acct}/app.bsky.feed.post/{op_id}'
        
        json_line = {
            'uri': repost_uri,
            'created-at': created_at,
            'reposted': reposted,
            'raw': {
                '$type': 'app.bsky.feed.repost',
                'subject': {
                    'cid': make_random_string(59),
                    'uri': reposted
                },
                'createdAt': created_at
            }
        }
        json_lines.append(json_line)


In [28]:
json.dump(json_lines, open('test_data/a.json', 'w'))

In [49]:
json_lines[0]

{'uri': 'at://did:plc:aaaaaaaaaaaaaaaaaaaaaaaa/app.bsky.feed.repost/7dNUywfuzS1QX',
 'created-at': '2023-06-17T04:21:04.873Z',
 'reposted': 'at://did:plc:abababababababababababab/app.bsky.feed.post/RxDbzmQTljVIx',
 'raw': {'$type': 'app.bsky.feed.repost',
  'subject': {'cid': 'oTO4LZGcCojzPqCtFsxC9sg3ziOF8LirLBmAZKNuPZRV39zHbTMXockNa5M',
   'uri': 'at://did:plc:abababababababababababab/app.bsky.feed.post/RxDbzmQTljVIx'},
  'createdAt': '2023-06-17T04:21:04.873Z'}}

In [41]:
from utils import *
def make_follows_in_range(n_follows, reposted_did, attention_broker, range_low, range_high):
    follows = []
    for i in range(n_follows):
        follower = 'did:plc:' + make_random_string(24)
        draw = random.uniform(0, 1)
        if draw > 0.25:
            utc_z = True
        elif draw > 0.1:
            utc_z = False
        else:
            utc_z = None

        follow_timestamp = make_random_date_str(range_low, range_high, utc_z=utc_z)

        follows.append({
            'from': follower, 
            'to': reposted_did, 
            'created_at': follow_timestamp,
        })
        draw2 = random.uniform(0, 1)
        if draw2 < 0.3:
            if draw2 < 0.15:
                follows.append({
                    'from': follower,
                    'to': attention_broker,
                    'created_at': make_random_date_str(range_low - dt.timedelta(days=20), range_low)
                })
            else:
                follows.append({
                    'from': follower,
                    'to': attention_broker,
                    'created_at': make_random_date_str(range_low, range_high + dt.timedelta(days=5))
                })

        if random.uniform(0, 1) < 0.1:
            follows.append({
                'from': follower,
                'to': None,
                'created_at': make_random_date_str(range_low, range_high)
            })
        if random.uniform(0, 1) < 0.3:
            follows.append({
                'from': attention_broker,
                'to': follower,
                'created_at': make_random_date_str(range_low - dt.timedelta(days=365), range_low)
            })
    
    return follows


reposts = json.load(open('./test_data/bsky_reposts/a.json'))
date_low = dt.datetime(year=2023, month=3, day=1, microsecond=0)
date_high = dt.datetime(year=2025, month=12, day=15, microsecond=0)
ab_did = 'did:plc:aaaaaaaaaaaaaaaaaaaaaaaa'

follows = []
for d in reposts:
    reposted_did = extract_did_from_uri(d['reposted'])
    repost_dt = dt.datetime.strptime(d['created-at'][:-1] + '000+00:00', '%Y-%m-%dT%H:%M:%S.%f%z')
    one_day_before_repost = repost_dt - dt.timedelta(hours=24)
    one_day_after_repost = repost_dt + dt.timedelta(hours=24)
    fourteen_days_before_repost = repost_dt - dt.timedelta(hours=24 * 14)
    fourteen_days_after_repost = repost_dt + dt.timedelta(hours=24 * 14)

    follows.extend(make_follows_in_range(5, reposted_did, ab_did, one_day_before_repost, repost_dt))
    follows.extend(make_follows_in_range(10, reposted_did, ab_did,repost_dt, one_day_after_repost))
    follows.extend(make_follows_in_range(5, reposted_did, ab_did, one_day_after_repost, fourteen_days_after_repost))
    follows.extend(make_follows_in_range(5, reposted_did, ab_did, fourteen_days_before_repost, one_day_before_repost))
    follows.extend(make_follows_in_range(5, reposted_did, ab_did, fourteen_days_after_repost, fourteen_days_after_repost + dt.timedelta(days=365)))
    follows.extend(make_follows_in_range(5, reposted_did, ab_did, fourteen_days_before_repost - dt.timedelta(days=365), fourteen_days_before_repost))

print(follows)


[{'from': 'did:plc:cLNpYqDy6dsF1nVQTVFa5zvX', 'to': 'did:plc:aaaaaaaaaaaaaaaaaaaaaaaa', 'created_at': '2023-10-03T21:38:13.498Z'}, {'from': 'did:plc:0R1IqLlYW5Dq33TIVniwmWbC', 'to': 'did:plc:aaaaaaaaaaaaaaaaaaaaaaaa', 'created_at': '2023-10-03T18:47:34.121Z'}, {'from': 'did:plc:YnJCJg8Dq0PzWgJxtn7VPTJ9', 'to': 'did:plc:aaaaaaaaaaaaaaaaaaaaaaaa', 'created_at': '2023-10-03T16:26:14.540Z'}, {'from': 'did:plc:YnJCJg8Dq0PzWgJxtn7VPTJ9', 'to': 'did:plc:aaaaaaaaaaaaaaaaaaaaaaaa', 'created_at': '2023-10-05T10:16:27.181Z'}, {'from': 'did:plc:fb1FaVV3a3typq6A3OkOwVtg', 'to': 'did:plc:aaaaaaaaaaaaaaaaaaaaaaaa', 'created_at': '2023-10-03T12:21:44.780Z'}, {'from': 'did:plc:fb1FaVV3a3typq6A3OkOwVtg', 'to': 'did:plc:aaaaaaaaaaaaaaaaaaaaaaaa', 'created_at': '2023-10-07T04:33:51.084Z'}, {'from': 'did:plc:1rGGdVd5OjKL9RhSNYucLhUy', 'to': 'did:plc:aaaaaaaaaaaaaaaaaaaaaaaa', 'created_at': '2023-10-03T07:29:26.284Z'}, {'from': 'did:plc:1rGGdVd5OjKL9RhSNYucLhUy', 'to': None, 'created_at': '2023-10-03T13:52:

In [42]:
import pandas as pd
df_fol = pd.DataFrame(follows)
df_fol.to_csv('./test_data/follows_sample.csv', index=False, header=None)